# Imports

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/zeguru/Projects/AI/tiny-text-transformers


In [13]:
from pathlib import Path

from src.task.config import ModelConfig
from src.task.model import load_model
from src.infra.tokenizer import CharacterTokenizer
from src.task.inference.inference import TextGenerator


# Load the model

In [14]:

checkpoint = "../checkpoints/tiny_poet_take2.pt"

TEXT_TOKENIZER_PATH = Path("../checkpoints/shakespear_stories_tokenizer.json")
tokenizer = CharacterTokenizer.load(
    TEXT_TOKENIZER_PATH
    )

# Conf

generator_config = ModelConfig(context_length=128, d_model=128, d_ff=512, num_layers=4, number_of_heads=4)


# Load model/checkpoint from file 
model = load_model(
    checkpoint_path=checkpoint,
    config=generator_config,
    vocab_size=tokenizer.vocab_size,
)


# generator
text_generator = TextGenerator(
    model=model,
    tokenizer=tokenizer,
    context_length=generator_config.context_length,
)

At TinyTransformerLM
number of heads:  4
multi head
number of heads:  4
multi head
number of heads:  4
multi head
number of heads:  4
multi head


# Bulk prompts
## Fixed length

In [15]:
import matplotlib.pyplot as plt

temperatures = [0.5, 0.7, 0.9, 1.1]

prompts = [
    "The boy",
    "She looked at",
    "One day",
    "The king said",
    "I think that",
]

for prompt in prompts:
    print("=" * 80)
    print(f"PROMPT: {prompt}")

    for temperature in temperatures:
        generated = text_generator.generate(
            prompt_text=prompt,
            max_new_tokens=100,
            temperature=temperature,
        )

        print()
        print(f"Temperature {temperature}:")
        print(generated)

PROMPT: The boy

Temperature 0.5:
The boy's time!

LEONTES:
I do, my lord, sir.

DUKE VINCENTIO:
No, he is the former heard the fires of the 

Temperature 0.7:
The boy's head, and the markers showing the lazy and that had shame to see the stack and promised to find a

Temperature 0.9:
The boy think he fallen herself, and I, that Buckingham,
Thou out our split of joy'd centerst, running
And 

Temperature 1.1:
The boy? Phoebutch she should look to discove again, it now
Might himself wrongs; she may not. These be the
PROMPT: She looked at

Temperature 0.5:
She looked at the sun and said, "You can show it were magically looking when they looked around the boy find and 

Temperature 0.7:
She looked at the best day she could to come to be car the other appel. Suddenly, Jane saw that him and the lawye

Temperature 0.9:
She looked at the fun rabbit and gardened a place clean, and sometime away there always rest. He stopped to the t

Temperature 1.1:
She looked at the success. In sseech just co

In [ ]:
## Variable lenghts. 
### Does the model begin coherent then drifts, or whether it is incoherent from the first few words

In [16]:
prompts = [
    "The boy",
    "One day",
    "She looked at",
    "The king said",
]

temperatures = [0.5, 0.7, 0.9]
lengths = [20, 40, 80, 128]

for prompt in prompts:
    print("=" * 100)
    print(f"PROMPT: {prompt}")

    for temperature in temperatures:
        print(f"\n--- temperature={temperature} ---")

        for length in lengths:
            generated = text_generator.generate(
                prompt_text=prompt,
                max_new_tokens=length,
                temperature=temperature,
            )

            print(f"\n[{length} tokens]")
            print(generated)

PROMPT: The boy

--- temperature=0.5 ---

[20 tokens]
The boy was a big delight l

[40 tokens]
The boy's face, the great shall be the deliciou

[80 tokens]
The boy saw a little girl named Lily. She was so happy and something special and said, 

[128 tokens]
The boy said, "You can speak at here you have something big prince. It made you to go on a ball."

The little girl called Daisy and sai

--- temperature=0.7 ---

[20 tokens]
The boy's bow'd battless to

[40 tokens]
The boy saw a little girl who loved to ask his 

[80 tokens]
The boy said, "Yes, I can go for you out again."

Tom smiled and said, "You have fun. I

[128 tokens]
The boy was so excited and said, "I can I need to take the care of mine eyes, the fiery in your brother worship? When Granna took me wi

--- temperature=0.9 ---

[20 tokens]
The boy smiled around and n

[40 tokens]
The boy said, "Hi know a green careful in the m

[80 tokens]
The boy's majesty regulary!

The voice stometh asked the garden and very goodbye. So, B


In [ ]:
# UNK investigation

In [8]:

shakespear_corpus = "../data/tiny_shakespear.txt"
tinystories_corpus = "../data/tinystories_300k.txt"

checkpoint = "../checkpoints/mh_tiny_text_generator.pt"
shakespear_tokenizer = "../checkpoints/shakespeare_tokenizer.json"

tokenizer_path = Path(shakespear_tokenizer)
sp_data_path = Path(shakespear_corpus)
ts_data_path = Path(tinystories_corpus)

tokenizer = CharacterTokenizer.load(tokenizer_path)
print(f"Vocab size: {tokenizer.vocab_size}")

shakespear_text = sp_data_path.read_text(encoding="utf-8",)
tinystories_text = ts_data_path.read_text(encoding="utf-8",)

sp_text_length = len(shakespear_text)
print(f"Shakespear Characters: {sp_text_length:,}")

ts_text_length = len(tinystories_text)
print(f"Tiny Stories Characters: {ts_text_length:,}")


Vocab size: 67
Shakespear Characters: 1,115,394
Tiny Stories Characters: 299,987


In [11]:
combined_text = shakespear_text + "\n" + tinystories_text

tokenizer = CharacterTokenizer(combined_text)

print("Vocabulary size:", tokenizer.vocab_size)

Vocabulary size: 77


In [ ]:
# Better Tokenizer

In [12]:

TEXT_TOKENIZER_PATH = Path("../checkpoints/shakespear_stories_tokenizer.json")
tokenizer.save(TEXT_TOKENIZER_PATH)